# PSA Machine Translation — Week 3: Modeling with Transfer Learning

**Course:** DSA 4020A NLP · Semester Project · **Focus:** Sub-objective 2 (few-shot cross-lingual transfer)

This notebook consumes the **Week 2 outputs** (`data_processed/`) and delivers the Week 3 checklist:

- Experiment tracking (**MLflow**, offline-friendly; optional Weights & Biases)
- **Two** pretrained models with few-shot fine-tuning: **NLLB-200-distilled-600M** and **mT5-small**
- Low-resource handling: **encoder freezing** + few-shot subsampling (+ back-translation hook)
- **Ablation studies**: zero-shot vs few-shot, freeze vs full fine-tune, per-domain
- Saved **checkpoints & logs**, documented **hyperparameters**, and an **initial performance summary**
- A working **inference demo** (notebook function + a `translate_psa.py` CLI)

**Design rationale (important):** NLLB-200 *does not include Ekegusii*, so we use it as a strong
baseline for the supported **English↔Kiswahili** directions. **mT5-small is text-to-text and
language-agnostic**, so it is our workhorse for the low-resource **Ekegusii** directions (the
project's headline goal). Using both satisfies "≥2 models" and directly demonstrates the
low-resource transfer story.

> **Runtime:** built for **Google Colab (GPU)**. Training cells download model weights from
> Hugging Face and need a GPU; the data/metric/inference plumbing runs on CPU. Set `DRY_RUN = True`
> first to smoke-test the whole pipeline in ~1 minute before committing to a full run.

## 0 · Environment, dependencies & hardware

The first cell **auto-installs** anything missing (Colab or local), importing only what isn't
already present — so you never hit a `ModuleNotFoundError`. If Colab says "Restart session" after
installing, do it, then run this cell again (it will skip the install and continue).

In [ ]:
# --- auto-install missing dependencies into THIS kernel -----------------------
import importlib.util, subprocess, sys

REQUIREMENTS = {
    "transformers": "transformers>=4.41",
    "datasets":     "datasets>=2.19",
    "accelerate":   "accelerate>=0.30",
    "sacrebleu":    "sacrebleu",
    "sentencepiece":"sentencepiece",
    "mlflow":       "mlflow",
    "evaluate":     "evaluate",
}
missing = [spec for mod, spec in REQUIREMENTS.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Done. If Colab shows a 'Restart session' prompt, restart and re-run this cell.")
else:
    print("All dependencies already present.")

In [ ]:
import os, time, json, random, math, glob
from pathlib import Path
import numpy as np
import pandas as pd

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Training will be slow — use DRY_RUN=True or a Colab GPU runtime "
          "(Runtime -> Change runtime type -> GPU).")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1 · Configuration

In [ ]:
# ---- Paths (Week 2 produced these) ------------------------------------------
PROC_DIR   = Path("data_processed")          # from the Week 2 notebook
MODEL_DIR  = Path("models_week3");  MODEL_DIR.mkdir(exist_ok=True)
MLRUNS_DIR = Path("mlruns");        MLRUNS_DIR.mkdir(exist_ok=True)
RESULTS_DIR= Path("results_week3"); RESULTS_DIR.mkdir(exist_ok=True)

# ---- What to translate ------------------------------------------------------
# NLLB-200 language codes; Ekegusii is NOT in NLLB (hence None).
NLLB_CODE = {"English": "eng_Latn", "Kiswahili": "swh_Latn", "Ekegusii": None}

# Directions available from Week 2 (slug = "<Src>_to_<Tgt>")
NLLB_DIRECTIONS = ["English_to_Kiswahili", "Kiswahili_to_English"]          # NLLB-supported
MT5_DIRECTIONS  = ["English_to_Kiswahili", "Kiswahili_to_English",
                   "English_to_Ekegusii",  "Ekegusii_to_English",
                   "Kiswahili_to_Ekegusii", "Ekegusii_to_Kiswahili"]        # incl. low-resource

# ---- Models -----------------------------------------------------------------
NLLB_NAME = "facebook/nllb-200-distilled-600M"
MT5_NAME  = "google/mt5-small"

# ---- Training hyperparameters (documented for the report) -------------------
DRY_RUN       = True     # <- keep True for the first pass (tiny subset, 1 epoch)
MAX_LEN       = 128
FEWSHOT_N     = 2000     # cap train examples per direction (few-shot / low-resource setting)
FREEZE_ENCODER= True     # low-resource technique: freeze encoder to curb overfitting

MT5_CFG  = dict(lr=1e-3, epochs=3, batch=8, optim="adafactor")   # mT5 likes Adafactor + higher LR
NLLB_CFG = dict(lr=3e-5, epochs=3, batch=8, optim="adamw_torch") # NLLB fine-tunes at a low LR

# ---- Experiment tracking ----------------------------------------------------
USE_WANDB = False        # set True and `wandb login` on Colab to also log to W&B
REPORT_TO = ["mlflow"] + (["wandb"] if USE_WANDB else [])

if DRY_RUN:
    FEWSHOT_N = 40
    MT5_CFG["epochs"] = 1; NLLB_CFG["epochs"] = 1
    print(">>> DRY_RUN active: tiny data + 1 epoch (plumbing test only).")

## 1b · Put `data_processed/` in place (Colab upload **or** local folder)

This cell makes the notebook work in either environment:

- **Local Jupyter** — if `data_processed/` already sits next to the notebook, it just verifies it.
- **Google Colab** — it prompts you to upload `data_processed.zip` and unzips it for you.

It also copes with either way of zipping on Windows: zipping the **folder**
(`data_processed/...` inside the zip) or zipping the **18 files** directly (loose in the zip).

In [ ]:
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Environment:", "Google Colab" if IN_COLAB else "local Jupyter")

SLUGS = ["English_to_Kiswahili", "Kiswahili_to_English",
         "English_to_Ekegusii",  "Ekegusii_to_English",
         "Kiswahili_to_Ekegusii", "Ekegusii_to_Kiswahili"]

def _has_data():
    return Path("data_processed").exists() and bool(list(Path("data_processed").glob("*_to_*.csv")))

def _normalise_layout():
    # After extraction, make sure the 18 CSVs end up under ./data_processed/
    if _has_data():
        return
    # Case A: zip contained a single nested folder holding the CSVs -> rename it.
    for cand in Path(".").iterdir():
        if cand.is_dir() and cand.name != "data_processed" and list(cand.glob("*_to_*.csv")):
            cand.rename("data_processed"); print("Renamed", cand.name, "-> data_processed"); return
    # Case B: CSVs extracted loose into the working dir -> move them into data_processed/.
    loose = list(Path(".").glob("*_to_*.csv"))
    if loose:
        Path("data_processed").mkdir(exist_ok=True)
        for f in loose:
            f.rename(Path("data_processed") / f.name)
        print(f"Moved {len(loose)} loose CSVs into data_processed/")

def _unzip_local():
    for z in sorted(Path(".").glob("data_processed*.zip")):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(".")
        print("Unzipped", z.name)
    _normalise_layout()

# 1) already present? 2) a zip already in the folder? 3) on Colab, ask for upload.
if not _has_data():
    _unzip_local()
if not _has_data() and IN_COLAB:
    from google.colab import files
    print("\nUpload your data_processed.zip now:")
    files.upload()
    _unzip_local()

# ---- verify ----
expected = [f"data_processed/{s}.{sp}.csv" for s in SLUGS for sp in ("train", "dev", "test")]
missing  = [f for f in expected if not Path(f).exists()]
print(f"\ndata_processed/: {len(expected) - len(missing)}/{len(expected)} expected files present")
if missing and _has_data():
    print("Some files missing (Ekegusii splits can be legitimately absent if you have no Ekegusii data):")
    for m in missing:
        print("  -", m)
elif not _has_data():
    raise FileNotFoundError(
        "No data_processed/ found. Locally: place the folder next to this notebook, or set "
        "PROC_DIR to its path in the config cell. On Colab: upload data_processed.zip when prompted.")
else:
    print("All 18 files present — ready to train.")

## 2 · Load the Week 2 data into Hugging Face datasets

Each Week 2 file (`<Src>_to_<Tgt>.<split>.csv`) already holds one direction with columns
`src_text, tgt_text, src_code, tgt_code, PSA_ID, Domain`. We wrap each direction in a
`DatasetDict` and, in the few-shot / low-resource setting, cap the training size to `FEWSHOT_N`.

In [ ]:
from datasets import Dataset, DatasetDict

def load_direction(slug, few_shot_n=None):
    dsd = {}
    for sp in ("train", "dev", "test"):
        p = PROC_DIR / f"{slug}.{sp}.csv"
        if p.exists():
            df = pd.read_csv(p).dropna(subset=["src_text", "tgt_text"])
            if sp == "train" and few_shot_n:
                df = df.sample(min(len(df), few_shot_n), random_state=SEED).reset_index(drop=True)
            dsd[sp] = Dataset.from_pandas(df, preserve_index=False)
    return DatasetDict(dsd)

# Sanity: how many pairs per direction / split?
rows = []
for slug in MT5_DIRECTIONS:
    d = load_direction(slug)
    rows.append({"direction": slug, **{sp: (len(d[sp]) if sp in d else 0)
                                        for sp in ("train", "dev", "test")}})
avail = pd.DataFrame(rows).set_index("direction")
print(avail)

## 3 · Evaluation metrics (BLEU + chrF++)

We report **BLEU** and **chrF++** via `sacrebleu`. chrF++ is character-n-gram based and is the more
reliable signal for morphologically rich Bantu languages (Kiswahili, Ekegusii), so we select the
best checkpoint on chrF. (COMET and full human evaluation come in Week 4.)

In [ ]:
import sacrebleu

def corpus_scores(preds, refs):
    preds = [p.strip() for p in preds]
    refs  = [r.strip() for r in refs]
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score   # word_order=2 => chrF++
    return {"bleu": round(bleu, 2), "chrf": round(chrf, 2)}

# Quick self-test of the metric on a trivial example
_demo = corpus_scores(["habari ya asubuhi"], ["habari ya asubuhi"])
print("metric self-test (perfect match):", _demo)

## 4 · Experiment tracking (MLflow)

In [ ]:
import mlflow
# Recent MLflow deprecated the plain file store; sqlite works everywhere (incl. Colab).
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("psa-mt-week3")
print("MLflow logging to sqlite:///mlflow.db")
# View later with:  !mlflow ui --backend-store-uri sqlite:///mlflow.db

## 5 · Shared model utilities

One place for the pieces both models share: encoder freezing (the low-resource trick),
a batched generation helper for zero-shot evaluation, and a `HF Trainer` factory.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)

def freeze_encoder(model):
    # Low-resource technique: freeze encoder weights so few examples can't overfit them.
    n = 0
    for p in model.get_encoder().parameters():
        p.requires_grad = False; n += p.numel()
    print(f"  froze encoder ({n/1e6:.1f}M params)")

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def generate_translations(model, tokenizer, texts, model_type,
                          src_lang, tgt_lang, batch_size=16, max_len=MAX_LEN):
    # Batched inference used for zero-shot eval and the demo. Returns list[str].
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i + batch_size])
        if model_type == "nllb":
            tokenizer.src_lang = NLLB_CODE[src_lang]
            enc = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_len).to(model.device)
            forced = tokenizer.convert_tokens_to_ids(NLLB_CODE[tgt_lang])
            gen = model.generate(**enc, forced_bos_token_id=forced, max_length=max_len)
        else:  # mt5
            prompt = [f"translate {src_lang} to {tgt_lang}: {t}" for t in batch]
            enc = tokenizer(prompt, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_len).to(model.device)
            gen = model.generate(**enc, max_length=max_len)
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return [o.strip() for o in out]

def make_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple): preds = preds[0]
        preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        dpred = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        dref  = tokenizer.batch_decode(labels, skip_special_tokens=True)
        return corpus_scores(dpred, dref)
    return compute_metrics

import inspect
def build_seq2seq_trainer(model, args, train_ds, eval_ds, collator, compute_metrics, tokenizer):
    # Recent transformers renamed Trainer's `tokenizer=` arg to `processing_class=`.
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
              data_collator=collator, compute_metrics=compute_metrics)
    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    kw["processing_class" if "processing_class" in params else "tokenizer"] = tokenizer
    return Seq2SeqTrainer(**kw)

def trainer_tokenizer(tr):
    # Fetch the tokenizer back off a Trainer across versions.
    return getattr(tr, "tokenizer", None) or getattr(tr, "processing_class", None)

## 6 · Model A — NLLB-200-distilled (English ↔ Kiswahili)

Two arms, which together form the **zero-shot vs few-shot** ablation:

1. **Zero-shot baseline** — the off-the-shelf model, no training. This is our reference point.
2. **Few-shot fine-tune** — same model, fine-tuned on the (capped) PSA training set, optionally with
   a frozen encoder.

In [ ]:
RESULTS = []   # collected across all experiments -> ablation table in section 8

def eval_nllb_zero_shot(direction):
    src, tgt = direction.split("_to_")
    dsd = load_direction(direction)
    if "test" not in dsd or len(dsd["test"]) == 0:
        print("  no test data for", direction); return
    tok = AutoTokenizer.from_pretrained(NLLB_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_NAME).to(DEVICE)
    preds = generate_translations(model, tok, dsd["test"]["src_text"], "nllb", src, tgt)
    sc = corpus_scores(preds, dsd["test"]["tgt_text"])
    RESULTS.append({"model": "NLLB-distilled", "direction": direction,
                    "setting": "zero-shot", **sc, "n_test": len(dsd["test"])})
    with mlflow.start_run(run_name=f"nllb-zeroshot-{direction}"):
        mlflow.log_params({"model": NLLB_NAME, "setting": "zero-shot", "direction": direction})
        mlflow.log_metrics(sc)
    print(f"  NLLB zero-shot {direction}: {sc}")
    del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None

for d in NLLB_DIRECTIONS:
    eval_nllb_zero_shot(d)

In [ ]:
def preprocess_nllb(tokenizer, src, tgt):
    tokenizer.src_lang = NLLB_CODE[src]
    tokenizer.tgt_lang = NLLB_CODE[tgt]
    def fn(batch):
        enc = tokenizer(batch["src_text"], text_target=batch["tgt_text"],
                        max_length=MAX_LEN, truncation=True)
        return enc
    return fn

def finetune_nllb(direction, cfg=NLLB_CFG, freeze=FREEZE_ENCODER):
    src, tgt = direction.split("_to_")
    dsd = load_direction(direction, few_shot_n=FEWSHOT_N)
    if "train" not in dsd or len(dsd["train"]) == 0:
        print("  no train data for", direction); return None
    tok = AutoTokenizer.from_pretrained(NLLB_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_NAME).to(DEVICE)
    if freeze: freeze_encoder(model)

    enc = dsd.map(preprocess_nllb(tok, src, tgt), batched=True,
                  remove_columns=dsd["train"].column_names)
    out_dir = MODEL_DIR / f"nllb_{direction}"
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch"], per_device_eval_batch_size=cfg["batch"],
        num_train_epochs=cfg["epochs"], weight_decay=0.0, optim=cfg["optim"],
        predict_with_generate=True, generation_max_length=MAX_LEN,
        fp16=(DEVICE == "cuda"), eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="chrf",
        greater_is_better=True, logging_steps=25, report_to=REPORT_TO,
        run_name=f"nllb-fewshot-{direction}",
    )   # older transformers: rename eval_strategy -> evaluation_strategy
    trainer = build_seq2seq_trainer(
        model=model, args=args, train_ds=enc["train"],
        eval_ds=enc.get("dev", enc["train"]),
        collator=DataCollatorForSeq2Seq(tok, model=model),
        compute_metrics=make_compute_metrics(tok), tokenizer=tok,
    )
    t0 = time.time(); trainer.train(); mins = (time.time() - t0) / 60

    test = enc.get("test")
    sc = trainer.evaluate(test) if test is not None else {}
    sc = {"bleu": round(sc.get("eval_bleu", float("nan")), 2),
          "chrf": round(sc.get("eval_chrf", float("nan")), 2)}
    RESULTS.append({"model": "NLLB-distilled", "direction": direction,
                    "setting": f"few-shot{'+freeze' if freeze else ''}",
                    **sc, "n_test": len(dsd.get("test", []))})
    trainer.save_model(str(out_dir)); tok.save_pretrained(str(out_dir))
    print(f"  NLLB few-shot {direction}: {sc} | {mins:.1f} min | "
          f"trainable={count_trainable(model)/1e6:.1f}M")
    return trainer

for d in NLLB_DIRECTIONS:
    finetune_nllb(d)

## 7 · Model B — mT5-small (all directions, incl. low-resource Ekegusii)

mT5 is pretrained on span-corruption, not translation, so it is used **fine-tuned** (a zero-shot
mT5 arm is not meaningful — a useful contrast to note in the report). Because it is text-to-text,
it handles **Ekegusii** with no predefined language code — this is what makes the low-resource
directions possible.

In [ ]:
def preprocess_mt5(tokenizer, src, tgt):
    def fn(batch):
        prompt = [f"translate {src} to {tgt}: {t}" for t in batch["src_text"]]
        enc = tokenizer(prompt, max_length=MAX_LEN, truncation=True)
        lab = tokenizer(text_target=batch["tgt_text"], max_length=MAX_LEN, truncation=True)
        enc["labels"] = lab["input_ids"]
        return enc
    return fn

def finetune_mt5(direction, cfg=MT5_CFG, freeze=FREEZE_ENCODER):
    src, tgt = direction.split("_to_")
    dsd = load_direction(direction, few_shot_n=FEWSHOT_N)
    if "train" not in dsd or len(dsd["train"]) == 0:
        print("  no train data for", direction); return None
    tok = AutoTokenizer.from_pretrained(MT5_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME).to(DEVICE)
    if freeze: freeze_encoder(model)

    enc = dsd.map(preprocess_mt5(tok, src, tgt), batched=True,
                  remove_columns=dsd["train"].column_names)
    out_dir = MODEL_DIR / f"mt5_{direction}"
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch"], per_device_eval_batch_size=cfg["batch"],
        num_train_epochs=cfg["epochs"], weight_decay=0.0, optim=cfg["optim"],
        predict_with_generate=True, generation_max_length=MAX_LEN,
        fp16=False,  # mT5 is numerically unstable in fp16; keep fp32 (or use bf16 on Ampere+)
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="chrf", greater_is_better=True,
        logging_steps=25, report_to=REPORT_TO, run_name=f"mt5-fewshot-{direction}",
    )
    trainer = build_seq2seq_trainer(
        model=model, args=args, train_ds=enc["train"],
        eval_ds=enc.get("dev", enc["train"]),
        collator=DataCollatorForSeq2Seq(tok, model=model),
        compute_metrics=make_compute_metrics(tok), tokenizer=tok,
    )
    t0 = time.time(); trainer.train(); mins = (time.time() - t0) / 60
    test = enc.get("test")
    sc = trainer.evaluate(test) if test is not None else {}
    sc = {"bleu": round(sc.get("eval_bleu", float("nan")), 2),
          "chrf": round(sc.get("eval_chrf", float("nan")), 2)}
    RESULTS.append({"model": "mT5-small", "direction": direction,
                    "setting": f"few-shot{'+freeze' if freeze else ''}",
                    **sc, "n_test": len(dsd.get("test", []))})
    trainer.save_model(str(out_dir)); tok.save_pretrained(str(out_dir))
    print(f"  mT5 {direction}: {sc} | {mins:.1f} min")
    return trainer

mt5_trainers = {d: finetune_mt5(d) for d in MT5_DIRECTIONS}

## 8 · Ablation studies & initial performance summary

Three comparisons the rubric asks for:

- **Zero-shot vs few-shot** — already collected for NLLB (English↔Kiswahili).
- **Freeze vs full fine-tune** — re-run one direction with the encoder unfrozen and compare.
- **Per-domain** — break the best model's test score down by PSA domain (domain adaptation view).

In [ ]:
# 8a. Freeze vs full fine-tune, on one representative direction
ABLATION_DIR = "English_to_Kiswahili"
if not DRY_RUN:
    finetune_mt5(ABLATION_DIR, freeze=False)   # adds a "few-shot" (no +freeze) row for contrast

# 8b. Assemble the results table
res = pd.DataFrame(RESULTS)
if len(res):
    res = res[["model", "direction", "setting", "bleu", "chrf", "n_test"]]
    res = res.sort_values(["direction", "model", "setting"]).reset_index(drop=True)
display(res)
res.to_csv(RESULTS_DIR / "performance_summary.csv", index=False)

In [ ]:
# 8c. Per-domain breakdown for the best mT5 direction we trained
def per_domain_eval(direction):
    tr = mt5_trainers.get(direction)
    if tr is None: return None
    dsd = load_direction(direction)
    if "test" not in dsd: return None
    test = dsd["test"].to_pandas()
    src, tgt = direction.split("_to_")
    preds = generate_translations(tr.model, trainer_tokenizer(tr), test["src_text"].tolist(),
                                  "mt5", src, tgt)
    test["pred"] = preds
    out = []
    for dom, g in test.groupby("Domain"):
        sc = corpus_scores(g["pred"].tolist(), g["tgt_text"].tolist())
        out.append({"direction": direction, "domain": dom, **sc, "n": len(g)})
    return pd.DataFrame(out)

pd_res = per_domain_eval("English_to_Kiswahili")
if pd_res is not None:
    display(pd_res)
    pd_res.to_csv(RESULTS_DIR / "per_domain_english_kiswahili.csv", index=False)

## 9 · Save logs & summary

In [ ]:
summary = {
    "device": DEVICE, "dry_run": DRY_RUN, "few_shot_n": FEWSHOT_N,
    "freeze_encoder": FREEZE_ENCODER, "max_len": MAX_LEN,
    "mt5_cfg": MT5_CFG, "nllb_cfg": NLLB_CFG,
    "results": RESULTS,
}
with open(RESULTS_DIR / "week3_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:")
for p in sorted(RESULTS_DIR.glob("*")): print("  -", p)
print("Checkpoints under:", MODEL_DIR.resolve())
print("MLflow runs      : sqlite:///mlflow.db  (mlflow ui --backend-store-uri sqlite:///mlflow.db)")

## 10 · Inference demo (Success criterion: a working translation demo)

A single `translate()` entry point that loads a saved checkpoint and translates a PSA. The same
logic is written out to **`translate_psa.py`** so it doubles as the Week 3 command-line deliverable.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=4)
def _load(model_path, model_type):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    return tok, model

def translate(text, src_lang, tgt_lang, model_path, model_type="mt5"):
    tok, model = _load(model_path, model_type)
    return generate_translations(model, tok, [text], model_type, src_lang, tgt_lang)[0]

# Example (after training; picks the mT5 English->Kiswahili checkpoint if present)
_ckpt = MODEL_DIR / "mt5_English_to_Kiswahili"
if _ckpt.exists():
    demo = "Ministry of Health urges residents to complete their vaccination before Friday."
    print("EN :", demo)
    print("SW :", translate(demo, "English", "Kiswahili", str(_ckpt), "mt5"))
else:
    print("Train a model first (run sections 6–7), then re-run this cell.")

In [ ]:
%%writefile translate_psa.py
# Week 3 CLI deliverable: python translate_psa.py --text "..." --src English --tgt Kiswahili --model models_week3/mt5_English_to_Kiswahili
import argparse, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_CODE = {"English": "eng_Latn", "Kiswahili": "swh_Latn", "Ekegusii": None}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def translate(text, src, tgt, model_path, model_type):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    if model_type == "nllb":
        tok.src_lang = NLLB_CODE[src]
        enc = tok(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                             max_length=128)
    else:
        enc = tok(f"translate {src} to {tgt}: {text}", return_tensors="pt",
                  truncation=True, max_length=128).to(DEVICE)
        gen = model.generate(**enc, max_length=128)
    return tok.batch_decode(gen, skip_special_tokens=True)[0].strip()

if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Translate a PSA between English/Kiswahili/Ekegusii.")
    ap.add_argument("--text", required=True)
    ap.add_argument("--src", required=True)
    ap.add_argument("--tgt", required=True)
    ap.add_argument("--model", required=True, help="path to a saved checkpoint")
    ap.add_argument("--model_type", default="mt5", choices=["mt5", "nllb"])
    a = ap.parse_args()
    print(translate(a.text, a.src, a.tgt, a.model, a.model_type))

## 11 · Summary & hand-off to Week 4 (Evaluation & Deployment)

**Delivered this week**

- Two fine-tuned model families (NLLB-distilled for English↔Kiswahili, mT5-small for all
  directions including Ekegusii), tracked in MLflow with saved checkpoints under `models_week3/`.
- Ablations: zero-shot vs few-shot, freeze vs full fine-tune, and a per-domain breakdown.
- `results_week3/performance_summary.csv` (the initial performance summary) and a working
  `translate()` demo + `translate_psa.py` CLI.

**What the numbers will tell you**

- English↔Kiswahili should be strong (NLLB already knows both; fine-tuning adapts it to PSA style).
- Ekegusii will lag — expected, and exactly the low-resource gap the project studies. Note in the
  report whether encoder-freezing and the few-shot cap helped or hurt each direction.

**Week 4 next steps**

1. Add **COMET** (`unbabel-comet`) alongside BLEU/chrF, and run **human evaluation** (fluency /
   adequacy / cultural accuracy) on 100+ sentences using the Week 2 native-speaker subset.
2. **Error analysis** per domain and per direction; document limitations.
3. Wrap the best checkpoints in a **Streamlit/Gradio** app (input PSA → choose target language →
   translation + confidence + feedback form).
4. Finalise the GitHub repo: code, dataset link, notebooks, README, CC-BY license.

**Troubleshooting (Colab)**

- OOM → lower `batch`, keep `MAX_LEN=128`, or use `google/mt5-small` only.
- mT5 loss stuck at 0 / NaN → keep `fp16=False` (already set); mT5 is unstable in fp16.
- Slow → confirm GPU runtime; start with `DRY_RUN=True` to verify the pipeline end-to-end.